# TMDB Movies Data Pipeline and EDA

This notebook builds a data pipeline using the TMDB API, stores movie data in SQLite, and performs exploratory data analysis (EDA).

**Tasks:**
1. Build the pipeline: Fetch at least 20 movies from TMDB, store in SQLite.
2. Perform EDA: Load data, display first 5 rows, summary statistics, genre counts, and missing value analysis.

In [ ]:
import requests
import pandas as pd
import sqlite3

# --- CONFIG ---
TMDB_API_KEY = 'yourAPIkey'  # Replace with your TMDB API key
TMDB_DISCOVER_URL = 'https://api.themoviedb.org/3/discover/movie'

# --- FETCH MOVIES ---
def fetch_movies(api_key, n_movies=20):
    params = {
        'api_key': api_key,
        'sort_by': 'popularity.desc',
        'page': 1
    }
    movies = []
    while len(movies) < n_movies:
        response = requests.get(TMDB_DISCOVER_URL, params=params)
        data = response.json()
        movies.extend(data['results'])
        params['page'] += 1
        if params['page'] > data['total_pages']:
            break
    return movies[:n_movies]

# Fetch at least 20 movies
movies = fetch_movies(TMDB_API_KEY, n_movies=20)
df_movies = pd.DataFrame(movies)

# --- SAVE TO SQLITE ---
conn = sqlite3.connect('movies.db')
df_movies.to_sql('movies', conn, if_exists='replace', index=False)
conn.close()

print('Fetched and saved movies to SQLite.')

ProgrammingError: Error binding parameter 3: type 'list' is not supported

In [2]:
# --- LOAD FROM SQLITE ---
conn = sqlite3.connect('movies.db')
df = pd.read_sql('SELECT * FROM movies', conn)
conn.close()

df.head()

DatabaseError: Execution failed on sql 'SELECT * FROM movies': no such table: movies

In [ ]:
# --- SUMMARY STATISTICS ---
df.describe()

In [ ]:
# --- MOVIES PER GENRE ---
import collections
import ast

def extract_genres(genre_col):
    genres = []
    for entry in genre_col:
        try:
            genre_list = ast.literal_eval(entry) if isinstance(entry, str) else entry
            if isinstance(genre_list, list):
                genres.extend([g['name'] for g in genre_list if 'name' in g])
        except Exception:
            continue
    return genres

if 'genre_ids' in df.columns:
    print('Note: genre_ids column contains only IDs, not names.')
    genre_counts = df['genre_ids'].explode().value_counts()
    print(genre_counts)
elif 'genres' in df.columns:
    genres_flat = extract_genres(df['genres'])
    genre_counts = collections.Counter(genres_flat)
    print(pd.Series(genre_counts))
else:
    print('No genre information found.')

In [ ]:
# --- MISSING VALUES ---
df.isnull().sum()